# 복지시설 상담 데이터셋 (AIHub) — 탐색적 데이터 분석(EDA)

## 이 데이터는 무엇인가
- **출처**: AIHub「복지시설 상담데이터」(2021년 구축, 전북테크노파크 외)
- **성격**: ⚠️ 실제 통화 녹취가 **아님**. 사전에 작성된 상담 대본을
  크라우드워커가 **연기·낭독하여 녹음**한 *시뮬레이션* 데이터.
  → 사투리·욕설·민감내용 없음, 화자 20~60대만(미성년·고령 제외),
    간투어(아/어/음…) 전사에서 제거됨. "깨끗하지만 분포가 좁은" 데이터.
- **규모**: 전체 발화 약 2,276,065건 / wav 약 323GB / **16kHz mono**
- **3개 도메인**:
  - 대학병원 (HOS) — 진료안내·병원이용안내·민원
  - 광역이동지원센터 (MOB) — 상담·고객대응·민원
  - 정신건강복지센터 (MEN) — 정신건강상담·자살위기개입

## 디렉터리 큰 그림
01.데이터/
├─ 1.Training/
└─ 2.Validation/
├─ 원천데이터/ ← 음성 .wav
└─ 라벨링데이터/ ← 전사 .json (발화 1건 = JSON 1개)

> Zeroth와 달리 **음성과 라벨이 별도 트리로 분리**돼 있음.
> 두 트리는 평행 구조이며, 매칭은 파일명 stem 기준(자세한 내용 셀 3).

## 이 노트북에서 확인하는 것
1. 폴더 구조와 음성↔라벨 매칭 방법
2. 메타데이터 스키마와 화자/기기/길이 분포 (설계 편향 확인)
3. 전사 규칙 준수 여부 (숫자·라틴·간투어·특수기호)
4. 개인정보 비식별화 마커(`ㅇㅇㅇ` / `000`) 프로파일
5. 종결부호·문장 분할 특성
6. 비식별화 토큰의 **실제 음성 재생**

> **재사용**: 대부분의 셀은 상단 `LABEL_ROOT` 한 곳만 바꾸면
> `1.Training`에도 그대로 적용됩니다.

In [5]:
# ── 셀 2: 공통 설정 ───────────────────────────────────────────
import json, re, random, statistics as st
from pathlib import Path
from collections import Counter, defaultdict

# 경로: 이 두 개만 split에 맞춰 바꾸면 노트북 전체가 재사용됩니다
LABEL_ROOT = Path("/data/ASR/RAW/AIHub_WelfareCounsel/01.데이터/2.Validation/라벨링데이터")
AUDIO_ROOT = Path("/data/ASR/RAW/AIHub_WelfareCounsel/01.데이터/2.Validation/원천데이터")
SEED = 42
random.seed(SEED)

# 라벨 JSON 전체 목록 (파일을 열지 않으므로 빠름)
json_files = sorted(LABEL_ROOT.rglob("*.json"))
print(f"라벨 JSON 총 개수: {len(json_files):,}")

def load_label(p):
    """JSON 1건 → (전사 리스트, 메타데이터 dict). 스키마 변형에 방어적."""
    d = json.loads(Path(p).read_text(encoding="utf-8"))
    texts = [t.get("orgtext", "") for t in d.get("inputText", [])]
    meta  = (d.get("info") or [{}])[0].get("metadata", {})
    apath = (d.get("dialogs") or [{}])[0].get("audioPath", "")
    return texts, meta, apath

# 도메인 분류 (경로의 VL 접두어 → 실패 시 파일명 코드로 폴백)
def domain_of(p):
    s = str(p)
    for code, name in [("VL1","대학병원(HOS)"),("VL2","광역이동지원센터(MOB)"),
                       ("VL3","정신건강복지센터(MEN)")]:
        if code in s:
            return name
    stem = Path(p).stem
    for pre, name in [("HOS","대학병원(HOS)"),("MOB","광역이동지원센터(MOB)"),
                      ("MEN","정신건강복지센터(MEN)")]:
        if stem.startswith(pre):
            return name
    return "UNKNOWN"

dom_cnt = Counter(domain_of(p) for p in json_files)
print("\n[도메인 분포]")
for k, v in sorted(dom_cnt.items()):
    print(f"  {k:24s}: {v:>8,} ({v/max(len(json_files),1)*100:5.1f}%)")

라벨 JSON 총 개수: 223,546

[도메인 분포]
  광역이동지원센터(MOB)           :   82,079 ( 36.7%)
  대학병원(HOS)               :   65,251 ( 29.2%)
  정신건강복지센터(MEN)           :   76,216 ( 34.1%)


## 셀 3 — 폴더 계층과 음성↔라벨 매칭

라벨 트리는 카테고리 코드를 따라 내려갑니다:

라벨링데이터/
└─ VL1_01.대학병원/ … /01.검사/ ← 도메인 > category2 > category3
└─ HOS3003221/ ← 세션(콜) 단위 폴더
└─ HOS11300322132B001.json ← 발화 1건


**음성 매칭 시 주의 — JSON 안의 `audioPath`를 그대로 쓰면 안 됩니다.**
`audioPath`는 `Y:\03.원천데이터\...wav` 형태의 **윈도우 경로**라 이 서버에서
무효합니다. 대신 **파일명 stem**으로 매칭합니다:
`…B001.json` ↔ `…B001.wav`

파일명 끝 한 글자도 의미가 있습니다 — **A=상담사, B=고객** (가이드라인 기준,
실제 JSON의 `speaker_type`과 일치 확인됨).

아래 `find_wav()`는 3단계로 wav를 찾습니다:
① 라벨 트리 미러링(`라벨링데이터`→`원천데이터`) → ② audioPath의 상대경로
결합 → ③ stem 인덱스(최후 수단, 최초 1회만 전체 스캔).

In [7]:
# ── 셀 3: 음성↔라벨 매칭 함수 (뒤 음성재생 셀에서 재사용) ──────
_stem_index = {}   # stem→wav 캐시 (③에서 1회만 구축)

def find_wav(json_path, audio_path_field=""):
    """라벨 JSON 경로 → 실제 wav 경로(없으면 None). 윈도우 audioPath는 직접 안 씀."""
    json_path = Path(json_path)
    # ① 라벨 트리를 그대로 미러링 (두 트리가 평행하면 가장 빠름)
    mirror = Path(str(json_path).replace("라벨링데이터", "원천데이터")).with_suffix(".wav")
    if mirror.exists():
        return mirror
    # ② audioPath의 '원천데이터' 이후 상대경로를 AUDIO_ROOT에 결합
    if audio_path_field:
        norm = audio_path_field.replace("\\", "/")
        if "원천데이터" in norm:
            cand = AUDIO_ROOT / norm.split("원천데이터/", 1)[-1]
            if cand.exists():
                return cand
    # ③ 전체 wav를 stem으로 인덱싱 (느릴 수 있음, 최초 1회만)
    if not _stem_index:
        print("· wav 인덱스 구축 중... (원천데이터 전체 스캔, 최초 1회)")
        for w in AUDIO_ROOT.rglob("*.wav"):
            _stem_index.setdefault(w.stem, w)
        print(f"· 인덱싱 완료: {len(_stem_index):,} 개")
    return _stem_index.get(json_path.stem)

# 매칭 동작 점검: 라벨 일부를 샘플링해 wav를 찾을 수 있는지 비율 확인
probe = random.sample(json_files, min(200, len(json_files)))
found = sum(find_wav(p, load_label(p)[2]) is not None for p in probe)
print(f"\n매칭 점검: {len(probe)}건 중 {found}건 wav 발견 "
      f"({found/len(probe)*100:.0f}%)")

# 세션 폴더 한 개를 열어 A(상담사)/B(고객) 짝 구조 확인
sample_json = probe[0]
session_dir = sample_json.parent
stems = sorted(q.stem for q in session_dir.glob("*.json"))

# 파일명 끝 'A###' / 'B###'에서 화자 구분 (정규식으로 명확히)
re_spk = re.compile(r"([AB])\d{3}$")
spk_tag = Counter()
for s in stems:
    m = re_spk.search(s)
    if m:
        spk_tag["A(상담사)" if m.group(1) == "A" else "B(고객)"] += 1

print(f"\n예시 세션 폴더: …/{session_dir.name}  (발화 {len(stems)}건)")
print(f"  화자 구분(파일명 끝 A/B): {dict(spk_tag)}")

# 파일명 끝 글자 ↔ JSON speaker_type 일치 검증 (가이드라인: A=상담사, B=고객)
print("  파일명 A/B ↔ speaker_type 교차 검증:")
for s in stems[:6]:
    m = re_spk.search(s)
    if not m:
        continue
    _, meta, _ = load_label(session_dir / f"{s}.json")
    print(f"    {s[-4:]} → speaker_type='{meta.get('speaker_type','?')}'")

· wav 인덱스 구축 중... (원천데이터 전체 스캔, 최초 1회)
· 인덱싱 완료: 223,546 개

매칭 점검: 200건 중 200건 wav 발견 (100%)

예시 세션 폴더: …/MOB0007741  (발화 93건)
  화자 구분(파일명 끝 A/B): {'A(상담사)': 54, 'B(고객)': 39}
  파일명 A/B ↔ speaker_type 교차 검증:
    A001 → speaker_type='상담사'
    A002 → speaker_type='상담사'
    A003 → speaker_type='상담사'
    A004 → speaker_type='상담사'
    A005 → speaker_type='상담사'
    A006 → speaker_type='상담사'


## 셀 4 — 메타데이터 스키마와 분포

각 JSON의 `info[0].metadata`에 발화 단위 메타가 들어있습니다(가이드라인 1.3):

| 항목 | 의미 | 값 범위 |
|---|---|---|
| category1/2/3 | 복지시설 / 상담유형 / 상담주제 | 도메인별 코드맵 |
| speaker_type | 발화자 구분 | 고객, 상담사 |
| speaker_id | 발화자 ID | SPK#### |
| speaker_age / sex | 연령대 / 성별 | 20~60대 / 남,여 |
| sptime_all/start/end | 녹음 전체/시작/종료(초) | — |
| rec_device / place | 녹음 기기 / 장소 | 스마트폰·PC / 집 등 |

**이 셀에서 확인할 것**: 화자 구성이 가이드라인 설계
(여 80% / 30~40대 중심 / 실내·스마트폰)와 맞는지. 분포가 좁다는 건
ASR 일반화의 한계로 이어지므로 EDA 결론에 명시할 항목입니다.

> 표본 추출(`N_META`)로 분포를 추정합니다. 22만 건 전수가 필요하면
> `N_META = None`으로 두세요(느려짐).

In [8]:
# ── 셀 4: 메타데이터 분포 ─────────────────────────────────────
N_META = 8000   # 분포 추정 표본 (전수는 None)
scan = json_files if N_META is None else random.sample(json_files, min(N_META, len(json_files)))
print(f"분포 추정 표본: {len(scan):,}\n")

meta_cnt = {k: Counter() for k in
            ("spk", "sex", "age", "dev", "place", "cat1")}
spk_ids, durations = set(), []

for p in scan:
    _, m, _ = load_label(p)
    meta_cnt["spk"][m.get("speaker_type", "?")]  += 1
    meta_cnt["sex"][m.get("speaker_sex", "?")]   += 1
    meta_cnt["age"][m.get("speaker_age", "?")]   += 1
    meta_cnt["dev"][m.get("rec_device", "?")]    += 1
    meta_cnt["place"][m.get("rec_place", "?")]   += 1
    meta_cnt["cat1"][m.get("category1", "?")]    += 1
    if m.get("speaker_id"):
        spk_ids.add(m["speaker_id"])
    try:
        durations.append(float(m.get("sptime_all", "nan")))
    except ValueError:
        pass

def show(title, counter, pct=True):
    tot = sum(counter.values())
    print(f"[{title}]")
    for k, v in counter.most_common():
        bar = "█" * int(v / max(tot, 1) * 30)
        suffix = f" ({v/tot*100:4.1f}%)" if pct else ""
        print(f"  {str(k):10s} {v:>6,}{suffix} {bar}")
    print()

show("speaker_type 고객/상담사", meta_cnt["spk"])
show("speaker_sex 성별",        meta_cnt["sex"])
show("speaker_age 연령대",      meta_cnt["age"])
show("rec_device 녹음기기",     meta_cnt["dev"])
show("rec_place 녹음장소",      meta_cnt["place"])
show("category1 도메인",        meta_cnt["cat1"])

print(f"[고유 화자 수(표본 내)] {len(spk_ids):,} 명")
ds = [x for x in durations if x == x]
if ds:
    ds.sort()
    print(f"[발화 길이 sptime_all]")
    print(f"  평균 {st.mean(ds):.2f}s · 중앙 {st.median(ds):.2f}s · "
          f"범위 {min(ds):.2f}~{max(ds):.2f}s")
    print(f"  사분위 P25 {ds[len(ds)//4]:.2f}s · P75 {ds[3*len(ds)//4]:.2f}s")

분포 추정 표본: 8,000

[speaker_type 고객/상담사]
  고객          4,646 (58.1%) █████████████████
  상담사         3,354 (41.9%) ████████████

[speaker_sex 성별]
  여           6,993 (87.4%) ██████████████████████████
  남           1,007 (12.6%) ███

[speaker_age 연령대]
  40대         2,281 (28.5%) ████████
  30대         2,271 (28.4%) ████████
  50대         1,728 (21.6%) ██████
  60대         1,188 (14.8%) ████
  20대           532 ( 6.7%) █

[rec_device 녹음기기]
  스마트폰        7,367 (92.1%) ███████████████████████████
  PC            633 ( 7.9%) ██

[rec_place 녹음장소]
  집           8,000 (100.0%) ██████████████████████████████

[category1 도메인]
  광역이동지원센터    2,944 (36.8%) ███████████
  정신건강복지센터    2,808 (35.1%) ██████████
  대학병원        2,248 (28.1%) ████████

[고유 화자 수(표본 내)] 181 명
[발화 길이 sptime_all]
  평균 4.77s · 중앙 4.55s · 범위 2.12~15.35s
  사분위 P25 3.85s · P75 5.40s


## 셀 5 — 전사 규칙 준수 검증

가이드라인 2.3.3은 전사 표기 규칙을 명시합니다. 핵심은 **"없어야 정상"**인
항목들입니다 — 이것들이 실제로 비어 있는지 정량 확인합니다:

- **아라비아 숫자 없음**: 모두 한글로 ("오 대" / "다섯 대"), 십진 단위 띄어쓰기
- **라틴 문자 없음**: 외래어도 한글 음차 (케이비에스, 엠비씨)
- **간투어 없음**: 아/그/어/음/저 등 제거
- **특수기호 없음**: (), #, * → 괄호, 샵, 별 (한글로)

반대로 **있어야** 하는 것은 비식별화 마커 두 종류 (셀 6에서 상세):
- `ㅇㅇㅇ` — 이름·소속 등 문자 비식별화
- `000` — 번호·날짜 등 숫자 비식별화

> 누출률이 0%에 가까우면 정제 품질이 양호하다는 뜻.
> 0이 아니면 그 사례를 직접 봐야 하므로 예시도 함께 출력합니다.

In [9]:
# ── 셀 5: 전사 규칙 준수 검증 ─────────────────────────────────
N_RULE = 8000
scan = json_files if N_RULE is None else random.sample(json_files, min(N_RULE, len(json_files)))

PATTERNS = {
    "아라비아숫자(1-9)": re.compile(r"[1-9]"),   # 0은 PII마커와 겹쳐 제외
    "라틴문자":          re.compile(r"[A-Za-z]"),
    "특수기호":          re.compile(r"[()\[\]{}#*/@~^<>]"),
}
# 실단어와 안 겹치는 보수적 간투어 집합 (그/저/아 등 다의어 제외)
FILLERS = {"음", "으", "응", "저기", "어어", "에에"}

hit = Counter()
examples = defaultdict(list)
filler_hit = 0
n_text = 0

for p in scan:
    texts, _, _ = load_label(p)
    for txt in texts:
        txt = (txt or "").strip()
        if not txt:
            continue
        n_text += 1
        for name, rgx in PATTERNS.items():
            if rgx.search(txt):
                hit[name] += 1
                if len(examples[name]) < 8:
                    examples[name].append(txt[:70])
        toks = txt.replace(",", " ").split()
        if any(tok in FILLERS for tok in toks):
            filler_hit += 1
            if len(examples["간투어"]) < 8:
                examples["간투어"].append(txt[:70])

print(f"분석 발화: {n_text:,}\n")
print("[전사 규칙 — '없어야 정상'인 항목]")
for name in PATTERNS:
    c = hit[name]
    print(f"  {name:16s}: {c:>5,} 건 ({c/max(n_text,1)*100:5.2f}%)")
print(f"  {'간투어(보수적)':16s}: {filler_hit:>5,} 건 ({filler_hit/max(n_text,1)*100:5.2f}%)")

print("\n[누출 사례 예시] — 0%가 아니면 직접 확인")
for name in list(PATTERNS) + ["간투어"]:
    if examples[name]:
        print(f"\n  ── {name} ──")
        for ex in examples[name]:
            print(f"     {ex}")

분석 발화: 8,000

[전사 규칙 — '없어야 정상'인 항목]
  아라비아숫자(1-9)     :     1 건 ( 0.01%)
  라틴문자            :     1 건 ( 0.01%)
  특수기호            :     0 건 ( 0.00%)
  간투어(보수적)        :     5 건 ( 0.06%)

[누출 사례 예시] — 0%가 아니면 직접 확인

  ── 아라비아숫자(1-9) ──
     센터가 정신건강 상담을 한 지 20년 정도 됐는데요.

  ── 라틴문자 ──
     건강을 나누는 ooo 정신건강복지센터 상담사 ㅇㅇㅇ입니다.

  ── 간투어 ──
     저기 ㅇㅇㅇㅇ에 있는 무슨 나라였는데
     근데 저기 내 친구가 말하길
     응 잘 다녔는데 제일 먹어보고 싶었던
     응 되게 많아
     응 집에서 출발해서 가야돼요


In [14]:
# ── 셀: 비표준 문자 전수 점검 (잡음/특수 태그 탐지) ──────────
N_CHARSCAN = 30000   # 희소 태그까지 잡으려면 크게 (전수는 None)
scan = json_files if N_CHARSCAN is None else random.sample(json_files, min(N_CHARSCAN, len(json_files)))

# "정상"으로 간주할 문자: 한글, 공백, 종결/쉼표, 그리고 PII 마커(ㅇ은 한글에 포함, 0만 추가)
re_normal = re.compile(r"[가-힣\s.,?!0]")
odd_chars = Counter()       # 비표준 문자 빈도
odd_examples = defaultdict(list)
n_text = 0

for p in scan:
    texts, _, _ = load_label(p)
    for txt in texts:
        txt = (txt or "").strip()
        if not txt:
            continue
        n_text += 1
        leftover = re_normal.sub("", txt)   # 정상 문자 제거 후 남는 것
        for ch in set(leftover):
            odd_chars[ch] += 1
            if len(odd_examples[ch]) < 3:
                odd_examples[ch].append(txt[:70])

print(f"분석 발화: {n_text:,}")
print(f"비표준 문자 종류: {len(odd_chars)}\n")

print("[비표준 문자 빈도] — 한글·공백·문장부호·PII(ㅇ,0) 외에 등장하는 모든 문자")
for ch, c in odd_chars.most_common():
    name = f"U+{ord(ch):04X}"
    print(f"  {ch!r:6s} ({name}) : {c:>5,} 발화")

print("\n[예시]")
for ch, c in odd_chars.most_common(20):
    print(f"\n  ── {ch!r} ({c:,}건) ──")
    for ex in odd_examples[ch]:
        print(f"     {ex}")

분석 발화: 30,000
비표준 문자 종류: 13

[비표준 문자 빈도] — 한글·공백·문장부호·PII(ㅇ,0) 외에 등장하는 모든 문자
  'ㅇ'    (U+3147) : 1,719 발화
  'o'    (U+006F) :    10 발화
  '․'    (U+2024) :     1 발화
  'ㆍ'    (U+318D) :     1 발화
  "'"    (U+0027) :     1 발화
  '4'    (U+0034) :     1 발화
  '2'    (U+0032) :     1 발화
  '1'    (U+0031) :     1 발화
  '~'    (U+007E) :     1 발화
  'k'    (U+006B) :     1 발화
  '\\'   (U+005C) :     1 발화
  '>'    (U+003E) :     1 발화
  'ㄱ'    (U+3131) :     1 발화

[예시]

  ── 'ㅇ' (1,719건) ──
     저는 아직도 ㅇㅇㅇ가 없다는 사실을 믿을 수가 없어요.
     ㅇㅇㅇ 교수님이십니다.
     방문하실 센터는 ㅇㅇ 동 근처로 안내해드리면 될까요?

  ── 'o' (10건) ──
     건강을 나누는 ooo 정신건강복지센터 상담사 ㅇㅇㅇ입니다.
     건강을 나누는 ooo 정신건강복지센터 상담사 ㅇㅇㅇ입니다.
     딸은 ooo. 000 0000 0000 이요.

  ── '․' (1건) ──
     상기 명시된 사항과 대상자 확인업무 등을 위하여 수집․처리할 수 있습니다.

  ── 'ㆍ' (1건) ──
     다른 곳에서 몇 버ㆍ 상담을 받아보기는 했고요.

  ── "'" (1건) ──
     그렇게 잘 개선되어 가고 있다니 다행이에요'

  ── '4' (1건) ──
     단순 방문이 아닌, 24시간 상주하신다면 방문자가 아닌 보호자로 저희가 따로 출입증을 드릴거거든요.

  ── '2' (1건) ──
     단순 방문이 아닌, 24시간 상주하신다면 방문자가 아닌 보호자로 저희

## 셀 6 — 개인정보 비식별화 마커 프로파일

전사문에는 두 종류의 비식별화 마커가 있습니다(가이드라인 2.3.3 개인정보 작성 기준):

- **`ㅇㅇㅇ`** (한글 ㅇ 반복) — 이름·계정·소속 등 **문자** 정보
- **`000`** (숫자 0 반복) — 전화·주소·날짜 등 **숫자** 정보

**핵심 — 길이 보존 마스킹**: 마커의 길이가 원래 정보의 길이를 반영합니다.
`00시`(2자리=시각), `0000년 00월 00일`(연4·월2·일2), `000 0000 0000`(전화 3-4-4).
그리고 음성에서는 두 마커 모두 **"공"으로 발음**됩니다(자릿수 = "공" 반복 횟수).

> ⚠️ **ASR 타겟 설계 결정사항** (지금 결론내지 않고 근거만 축적):
> 마커를 단일 `<PII>` 토큰으로 뭉치면 길이 정보가 사라집니다.
> 음성-텍스트 정렬에서 마커 길이 = "공" 발음 길이가 대응하므로,
> ① 마커 원형 유지 ② "공"으로 정규화 ③ 길이 보존 특수토큰 중
> 무엇을 택할지는 이 대응관계를 살릴지의 문제입니다. (실제 음성은 셀 8에서 청취)

In [10]:
# ── 셀 6: PII 마커 프로파일 ──────────────────────────────────
N_PII = 8000
scan = json_files if N_PII is None else random.sample(json_files, min(N_PII, len(json_files)))

re_o    = re.compile(r"ㅇ{2,}")
re_zero = re.compile(r"0{2,}")

o_lens, zero_lens = Counter(), Counter()
both_in_one = 0
n_with_marker, n_text = 0, 0
marker_examples = []

for p in scan:
    texts, _, _ = load_label(p)
    for txt in texts:
        txt = (txt or "").strip()
        if not txt:
            continue
        n_text += 1
        os = re_o.findall(txt)
        zs = re_zero.findall(txt)
        for m in os:
            o_lens[len(m)] += 1
        for m in zs:
            zero_lens[len(m)] += 1
        if os or zs:
            n_with_marker += 1
        if os and zs:
            both_in_one += 1
            if len(marker_examples) < 8:
                marker_examples.append(txt[:75])

print(f"분석 발화: {n_text:,}")
print(f"마커 포함 발화: {n_with_marker:,} ({n_with_marker/max(n_text,1)*100:.2f}%)\n")

print("[ㅇ 마커 길이 분포] — 보통 2~3 (소속=2, 이름=3 경향)")
for k in sorted(o_lens):
    print(f"  ㅇ×{k}: {o_lens[k]:>5,} {'█'*int(o_lens[k]/max(sum(o_lens.values()),1)*30)}")

print("\n[0 마커 길이 분포] — 자릿수가 원래 숫자 길이를 반영")
for k in sorted(zero_lens):
    print(f"  0×{k}: {zero_lens[k]:>5,} {'█'*int(zero_lens[k]/max(sum(zero_lens.values()),1)*30)}")

print(f"\n한 발화에 ㅇ마커·0마커 동시 등장: {both_in_one:,} 건")
print("  예시 (둘 다 '공'으로 발음됨):")
for ex in marker_examples:
    print(f"     {ex}")

분석 발화: 8,000
마커 포함 발화: 591 (7.39%)

[ㅇ 마커 길이 분포] — 보통 2~3 (소속=2, 이름=3 경향)
  ㅇ×2:   229 ████████████
  ㅇ×3:   331 █████████████████
  ㅇ×4:     6 
  ㅇ×5:     1 

[0 마커 길이 분포] — 자릿수가 원래 숫자 길이를 반영
  0×2:   101 ██████████
  0×3:    64 ██████
  0×4:   118 ████████████
  0×5:     2 
  0×6:     6 

한 발화에 ㅇ마커·0마커 동시 등장: 27 건
  예시 (둘 다 '공'으로 발음됨):
     네 내일 자택에서 ㅇㅇ병원 00시에 예약하신 ㅇㅇㅇ고객님 맞으십니까?
     저는 ㅇㅇㅇ입니다. 연락처는 000 0000 0000이구요
     ㅇㅇㅇ, 000 0000 0000 이예요.
     ㅇㅇ시 ㅇㅇ동 000번지에서 고객님 댁으로
     0월 0일 오전 00시 ㅇㅇㅇ시 ㅇㅇㅇ구 000번지인 고객님 자택에서
     ㅇㅇ시 ㅇㅇ동 00지에 위치한 ㅇㅇ병원까지 접수해 드렸습니다.
     이름은 ㅇㅇㅇ고요. 생년월일은 00년 0월 0일이요.
     성함은 ㅇㅇㅇ 요. 연락처가 000 0000 0000 에요.


## 셀 7 — 종결부호와 문장 분할 특성

가이드라인 2.3.3 차항은 "문장이 끝나면 반드시 마침표/물음표/느낌표"를 요구합니다.
하지만 실제로는 종결부호 없는 발화가 상당수 존재합니다. **이것이 규칙 위반(누락)인지,
아니면 문장단위 자동분할의 자연스러운 결과인지**를 발화 길이와 교차해 판별합니다.

- 종결부호 **없음이 짧은 발화(1-5자)에 몰리면** → 추임새·응답형 미완성 발화
- 종결부호 **없음이 긴 발화(16자+)에 몰리면** → 긴 문장이 중간에서 잘린 토막

가이드라인 2.5.1.2는 긴 발화를 **문장단위로 자동 분할**(앞뒤 묵음 최대 1초)한다고
명시합니다. 만약 후자라면, 종결부호 없음은 결함이 아니라 분할 정책의 산물입니다.

> 시사점: 종결부호로 문장 경계를 잡는 전처리는 이 데이터에 부적합.
> 또한 느낌표가 거의 없는 것은 낭독 특성(감탄 억양 부재)입니다.

In [11]:
# ── 셀 7: 종결부호 × 발화 길이 ───────────────────────────────
N_END = 8000
scan = json_files if N_END is None else random.sample(json_files, min(N_END, len(json_files)))

re_end = re.compile(r"[.?!]$")
end_kinds = Counter()
noend_by_len = defaultdict(int)
end_by_len   = defaultdict(int)
n_text = 0

def len_bucket(n):
    if n <= 5:   return "1-5자"
    if n <= 15:  return "6-15자"
    if n <= 30:  return "16-30자"
    return "31자+"

for p in scan:
    texts, _, _ = load_label(p)
    for txt in texts:
        txt = (txt or "").strip()
        if not txt:
            continue
        n_text += 1
        b = len_bucket(len(txt))
        if re_end.search(txt):
            end_kinds[txt[-1]] += 1
            end_by_len[b] += 1
        else:
            noend_by_len[b] += 1

print(f"분석 발화: {n_text:,}\n")

print("[종결부호 종류]")
for k, v in end_kinds.most_common():
    print(f"  {k} : {v:>6,} ({v/max(n_text,1)*100:5.1f}%)")
tot_noend = sum(noend_by_len.values())
print(f"  종결부호 없음 : {tot_noend:>6,} ({tot_noend/max(n_text,1)*100:5.1f}%)")

print("\n[길이 버킷별 종결부호 유/무]  — 없음이 어디에 몰리는지")
print(f"  {'버킷':8s} {'있음':>8s} {'없음':>8s} {'없음비율':>8s}")
for b in ["1-5자", "6-15자", "16-30자", "31자+"]:
    e, n = end_by_len[b], noend_by_len[b]
    tot = e + n
    if tot:
        print(f"  {b:8s} {e:>8,} {n:>8,} {n/tot*100:>7.1f}%")

분석 발화: 8,000

[종결부호 종류]
  . :  4,389 ( 54.9%)
  ? :  1,193 ( 14.9%)
  종결부호 없음 :  2,418 ( 30.2%)

[길이 버킷별 종결부호 유/무]  — 없음이 어디에 몰리는지
  버킷             있음       없음     없음비율
  1-5자           52       32    38.1%
  6-15자       1,639      768    31.9%
  16-30자      3,038    1,445    32.2%
  31자+          853      173    16.9%


## 셀 8 — 비식별화 토큰의 실제 음성 청취

마커가 음성에서 정말 "공"으로 발음되는지, 그리고 **마커 길이 = "공" 반복 횟수**가
맞는지 직접 들어봅니다. 세 유형으로 나눠 재생합니다:

- **ㅇ 마커만** — 이름·소속 (`ㅇㅇㅇ` → "공공공")
- **0 마커만** — 번호·날짜 (`000 0000 0000` → "공"×11)
- **ㅇ+0 동시** — 둘 다 "공"으로 들리는지가 핵심 (예: `ㅇㅇㅇ ... 000 0000 0000`)

> 청취 확인 포인트
> 1. `ㅇㅇㅇ`과 `000`이 글자는 달라도 **같은 "공" 소리**인가
> 2. 자릿수만큼 "공"이 반복되는가 (`00시` → "공공 시")
> 3. 어긋나는 사례가 있으면 = 라벨 정렬 이슈 후보

In [13]:
# ── 셀 8: 비식별화 토큰 음성 재생 ────────────────────────────
import soundfile as sf
from IPython.display import Audio, HTML, display

N_EACH = 4   # 유형별 재생 개수

re_o    = re.compile(r"ㅇ{2,}")
re_zero = re.compile(r"0{2,}")

def highlight(txt):
    # ㅇ 마커와 0 마커를 한 번에 매칭해 각각 색칠 (이중 치환 방지)
    def repl(m):
        s = m.group()
        color = "#c0392b" if s[0] == "ㅇ" else "#2471a3"
        return f"<span style='color:{color};font-weight:700'>{s}</span>"
    return re.sub(r"ㅇ{2,}|0{2,}", repl, txt)

buckets = {"ㅇ 마커만 (문자 비식별화)": [],
           "0 마커만 (숫자 비식별화)": [],
           "ㅇ+0 동시 (둘 다 '공')": []}

pool = json_files[:]
random.shuffle(pool)
for p in pool:
    if all(len(v) >= N_EACH for v in buckets.values()):
        break
    texts, meta, apath = load_label(p)
    txt = " ".join(texts).strip()
    has_o, has_z = bool(re_o.search(txt)), bool(re_zero.search(txt))
    if not (has_o or has_z):
        continue
    key = ("ㅇ+0 동시 (둘 다 '공')" if (has_o and has_z)
           else "ㅇ 마커만 (문자 비식별화)" if has_o
           else "0 마커만 (숫자 비식별화)")
    if len(buckets[key]) >= N_EACH:
        continue
    wav = find_wav(p, apath)        # 셀 3에서 정의한 매칭 함수 재사용
    if wav is None:
        continue
    buckets[key].append((p.stem, txt, wav, meta))

for key, rows in buckets.items():
    display(HTML(
        f"<h3 style='margin:16px 0 2px'>▶ {key} — {len(rows)}건</h3>"
        f"<div style='color:#888;font-size:12px'>"
        f"<span style='color:#c0392b;font-weight:700'>ㅇㅇㅇ</span> · "
        f"<span style='color:#2471a3;font-weight:700'>000</span> "
        f"모두 음성에서 “공”으로 발음 (자릿수 = ‘공’ 반복 횟수)</div>"))
    if not rows:
        display(HTML("<i style='color:#999'>이 유형 wav 미발견 — AUDIO_ROOT 확인</i>"))
    for stem, txt, wav, meta in rows:
        try:
            data, sr = sf.read(str(wav))
        except Exception as e:
            display(HTML(f"<div style='color:#999'>{stem}: 읽기 실패 ({e})</div>"))
            continue
        dur = len(data) / sr
        info = (f"{meta.get('speaker_type','?')}/{meta.get('speaker_sex','?')}"
                f"/{meta.get('speaker_age','?')} · "
                f"{meta.get('category1','?')}>{meta.get('category2','?')}"
                f">{meta.get('category3','?')} · {sr/1000:.0f}kHz {dur:.2f}s")
        display(HTML(f"<div style='margin-top:10px'><b>{stem}</b> "
                     f"<span style='color:#888;font-size:12px'>({info})</span><br>"
                     f"<span style='font-size:15px'>전사&gt; {highlight(txt)}</span></div>"))
        display(Audio(data=data, rate=sr))